# Suite INS — Installs: schemas defined by transactions

A module defines its schemas in a module-scope `install(txn)` (docs/OBJECT_IDENTITY.md, Decided, 86;
[Installing schemas](../../docs/proposals/install/README.md)): its transaction upserts them, each schema object a symbol
of its meta-schema, named from the transaction (`txn.Schemas.OfObject`, 78), found by its name, else created, and its
spec written where it does not hold. Run again, an install writes nothing; another adds to what it finds. A schema with
no name of its own is defined in a spec where it is linked, `txn.Schemas.OfContainer(cb)`, which creates nothing.

As in Python, with the language's differences (docs/EQUIVALENCE.md): builders, the store and a transaction are `Proxy`
objects typed loosely, so callbacks are annotated `(t: any)`; ints are `bigint`s.

In [ ]:
import { Builders as B, JSON as Json, Mutations, Paths, Plain, Schemas } from "@mbse/schemas/Framework";
import { AttributeError, ValueError } from "@mbse/schemas/Framework/Errors";
import { repr } from "@mbse/schemas/Framework/Repr";
import { assert, entries, raises, same } from "./support.js";

/** crm.Person, who knows persons since a year; crm.Shape, a union; crm.Grid, a container. */
function crm(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("crm/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Person: S.OfObject, Knows: S.OfRelation, Shape: S.OfUnion, Grid: S.OfContainer })
    .spec([
      (_: any) => _.Knows.name("crm.Knows").links([(t: any) => t.name("knower"), (t: any) => t.name("known")]).properties({
        since: S.OfInt }),
      (_: any) => _.Person.name("crm.Person").ref(true).properties({ name: S.OfStr }).adjacencies({
        knows: (a: any) => a.relation([_.Knows]).me("knower") }).adjacencies([
        (a: any) => a.relation([_.Knows]).me("known")]),
      (_: any) => _.Shape.name("crm.Shape").branches([
        (t: any) => t.name("circle").schema([S.OfFloat]), (t: any) => t.name("label").schema([S.OfStr])]),
      (_: any) => _.Grid.name("crm.Grid").index([S.OfInt]).element([S.OfFloat]),
    ])]))]);
}

/** crm.Person's age and a parameter, its era, added to what `crm` defined. */
function aged(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("crm/2").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Person: S.OfObject })
    .spec([(_: any) => _.Person.name("crm.Person").properties({ age: S.OfInt }).parameters({ era: (t: any) => t })])]))]);
}

/** x.Item, whose tags are a list of text, a container with no name of its own, defined where it is linked. */
function tagged(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("tags/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Item: S.OfObject })
    .spec([(_: any) => _.Item.name("x.Item").ref(true).properties({
      tags: S.OfContainer((c: any) => c.index([S.OfInt]).element([S.OfStr])) })])]))]);
}

## INS-01 · A module's install defines its schemas, each kind a symbol of its meta-schema; run again, it writes nothing

In [ ]:
{
  const store: any = new B.Store();
  store.transaction(crm);
  const [Person, Knows] = [store.registered("crm.Person"), store.registered("crm.Knows")];
  assert(Person.ref && same([...Person.properties.keys()], ["name"]) && Person.properties.get("name").schema === store.Schemas.OfStr.as_schema());
  assert(same([...Knows.entry.links.keys()], ["knower", "known"]) && same([...Knows.entry.properties.keys()], ["since"]));
  assert(Person.adjacencies.get("knows").relation === Knows && Person.adjacencies.get("knows").me === "knower");
  assert(same(store.registered("crm.Shape").branches.map((b: any) => b.name), ["circle", "label"]));
  const grid = store.registered("crm.Grid");
  assert(grid.index === store.Schemas.OfInt.as_schema() && grid.element === store.Schemas.OfFloat.as_schema());
  const snapshot = Json.dumps(Plain.ToPlain(store).Snapshot() as never);
  store.transaction(crm); // run again: each found by its name, nothing written
  assert(Json.dumps(Plain.ToPlain(store).Snapshot() as never) === snapshot);
  const ada = store.transaction(() => store.crm.Person((c: any) => c.name("Ada").knows((a: any) => a.insert((e: any) =>
    e.known((b: any) => b.name("Bob")).since(2020n))))); // what it defined builds data
  assert(same(entries(Person, ada, "knows").map((e) => `${e.get("known").name}`), ["Bob"]));
}

## INS-02 · A schema with no name of its own, defined in a spec where it is linked: made with the link, found holding when run again; linked twice, refused

In [ ]:
{
  const store: any = new B.Store();
  const outside = store.transaction(() =>
    store.Schemas.OfContainer((c: any) => c.index([store.Schemas.OfInt]).element([store.Schemas.OfStr]))); // made
  store.transaction(tagged);
  const tags = store.registered("x.Item").properties.get("tags").schema;
  assert(tags !== outside && tags.name === null); // defined where it is linked, made with the link
  assert(tags.index === store.Schemas.OfInt.as_schema() && tags.element === store.Schemas.OfStr.as_schema());
  store.transaction(tagged); // run again: an equal one is linked there, so nothing is made
  assert(store.registered("x.Item").properties.get("tags").schema === tags);
  const twice = (txn: any): any => {
    const S = txn.Schemas;
    return txn.uid("twice/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
      .symbols({ Item: S.OfObject })
      .spec([(_: any) => ((texts: any) => _.Item.name("x.Twice").properties({ a: texts, b: texts }))(
        S.OfContainer((c: any) => c.index([S.OfInt]).element([S.OfStr])))])]))]);
  };
  raises(ValueError, () => store.transaction(twice), "an object a spec defines is linked where it is defined"); // one place
}

## INS-03 · An install adds to what another defined; a transaction given to an install names its own members, and the store's schemas besides

In [ ]:
{
  const store: any = new B.Store();
  store.transaction(crm);
  store.transaction(aged);
  assert(same([...store.registered("crm.Person").properties.keys()], ["name", "age"]));
  assert(same([...store.registered("crm.Person").parameters.keys()], ["era"])); // an entry given nothing defines what it links
  const paired = (txn: any): any => {
    const S = txn.Schemas;
    return txn.uid("pair/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
      .symbols({ Pair: S.OfUnion })
      .spec([(_: any) => _.Pair.name("x.Pair").branches([(t: any) => t.name("n").schema([S.OfInt, S.OfStr])])])]))]);
  };
  raises(TypeError, () => store.transaction(paired), "link 'schema' links one schema, got 2");
  const seen: [boolean, string][] = [];
  const looking = (txn: any): any => {
    seen.push([txn.Schemas.OfInt.as_schema() === store.Schemas.OfInt.as_schema(), repr(txn)]);
    raises(AttributeError, () => txn._hidden);
    assert(txn.then === undefined); // JavaScript's protocol probes: awaiting it does not throw
    return txn.uid("look/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
      .symbols({ Int: txn.Schemas.OfScalar }).spec([(_: any) => _.Int.name("Schemas.OfInt")])]))]); // the store's own, found
  };
  store.transaction(looking);
  assert(seen.length === 1 && seen[0]![0] && seen[0]![1] === "<transaction>");
}

## INS-04 · The framework installs itself: a store is made by reading the installs of Schemas, Paths and Mutations; each, run again as a transaction, writes nothing

In [ ]:
{
  const store: any = new B.Store(); // made by the bootstrap, reading each install through the builders
  assert(store.registered("Mutations.OfProperty.Set").properties.get("value").schema === store.registered("Mutations.Payload"));
  assert(store.registered("Schemas.OfObject").adjacencies.get("properties").relation === store.registered("Schemas.HasProperty"));
  const held = new Map((store.names() as string[]).map((name) => [name, store.registered(name)]));
  for (const module of [Schemas, Paths, Mutations]) {
    store.transaction(module.install); // each found by its name; one it would change is refused (the store's own)
  }
  assert(store.names().length === held.size && (store.names() as string[]).every((name) => store.registered(name) === held.get(name)));
}